Python 3.11.7

In [51]:
!pip install --upgrade pip
!pip install requests pandas nltk torch transformers scikit-learn tqdm contractions

In [52]:
import requests
import json
import pandas as pd
import re
import nltk
import contractions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from datetime import datetime, timedelta
import torch.nn.functional as F

lemmatizer = WordNetLemmatizer()
nltk.download('punkt')  
nltk.download('stopwords')   # 불용어 리스트(최초 1회만 다운로드 필요)
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/leejunho/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/leejunho/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/leejunho/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [53]:
nyse = 'AAPL'
start = '2024-12-01'
end = '2025-03-31'

for i in range (1, 46):
    if(i == 1):
        url = 'https://api-v2.deepsearch.com/v1/global-articles?symbols=NYSE:'+nyse+'&page='+str(i)+'&page_size=100&date_from='+start+'&date_to='+end+'&api_key=cffaa97f7c2f4ad0bd501ce7cd00b7e1'
        data = requests.get(url).json()
        title_summary_list = [(item["title"], item["summary"], item["published_at"]) for item in data["data"]]
        df_news = pd.DataFrame([
            {"Date": published_at, "title": title, "summary": summary}
            for title, summary, published_at in title_summary_list
        ])
    else:
        url = 'https://api-v2.deepsearch.com/v1/global-articles?symbols=NYSE:'+nyse+'&page='+str(i)+'&page_size=100&date_from='+start+'&date_to='+end+'&api_key=cffaa97f7c2f4ad0bd501ce7cd00b7e1'
        data = requests.get(url).json()
        title_summary_list = [(item["title"], item["summary"], item["published_at"]) for item in data["data"]]
        df_news_temp = pd.DataFrame([
            {"Date": published_at, "title": title, "summary": summary}
            for title, summary, published_at in title_summary_list
        ])
        df_news = pd.concat([df_news, df_news_temp], ignore_index=True)

df_news['Date'] = pd.to_datetime(df_news['Date']).dt.tz_convert('America/New_York')
df_news['Date'] = df_news['Date'].dt.strftime('%Y-%m-%d %H:%M:00')

# # 5. 결과 출력
df_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Date     4500 non-null   object
 1   title    4500 non-null   object
 2   summary  4500 non-null   object
dtypes: object(3)
memory usage: 105.6+ KB


In [54]:
df_news

,Date,title,summary
0,2025-03-31 19:30:00,Get Apple AirPods for Less Than They Were Over...,Apple AirPods and EarPods are currently availa...
1,2025-03-31 19:24:00,Mobile companies seek zero duty on US imports,"In New Delhi, mobile phone manufacturers, incl..."
2,2025-03-31 19:04:00,"Kessler Topaz Meltzer &amp; Check, LLP Reminds...","Kessler Topaz Meltzer & Check, LLP announces t..."
3,2025-03-31 17:00:00,Why the Magnificent Seven Stocks Just Had Thei...,"The Magnificent Seven, comprising major tech s..."
4,2025-03-31 17:00:00,Apple and the bigger fight over Indonesia’s bi...,Indonesia's recent conflict with Apple regardi...
...,...,...,...
4495,2025-01-01 05:19:00,Biggest court cases of 2025: From Diddy and Lu...,"In 2024, significant legal cases are set to un..."
4496,2025-01-01 05:06:00,10 Stock Market Predictions for 2025,"As we enter 2024, investors express gratitude ..."
4497,2025-01-01 04:15:00,A cybersecurity executive was pardoned by Dona...,"In 2020, Donald Trump pardoned cybersecurity e..."
4498,2025-01-01 04:15:00,A cybersecurity executive was pardoned by Dona...,"In December 2020, Donald Trump pardoned Chris ..."


In [55]:
api_key = "7A2qEj2N1XlxkYJogRqKFtkBWUkr32UQ"
url = f"https://api.polygon.io/v2/aggs/ticker/{nyse}/range/1/minute/{start}/{end}?limit=50000&apiKey={api_key}"

response = requests.get(url)
data = response.json()
if "results" in data:
    df = pd.DataFrame(data["results"])
    df["t"] = pd.to_datetime(df["t"], unit="ms")
    df.rename(columns={"t":"Time","o": "Open", "h": "High", "l": "Low", "c": "Close", "v": "Volume"}, inplace=True)
    df["Change"] = df["Close"] - df["Open"]
    df = df[["Time","Open", "High", "Low", "Close", "Volume", "Change"]]
    df_news['Date'] = pd.to_datetime(df_news['Date'])
    df = pd.merge(df, df_news, left_on="Time", right_on="Date", how="left")
    df = df.drop(columns=["Date"])
    #title과 summary가 NaN인 행을 삭제 한다.
    df = df.dropna(subset=["title", "summary"])
else:
    print("Error: ", data["error"])
# # csv로 저장
df.to_csv("AAPL_2025.csv", index=False)

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1144 entries, 14316 to 50051
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Time     1144 non-null   datetime64[ns]
 1   Open     1144 non-null   float64       
 2   High     1144 non-null   float64       
 3   Low      1144 non-null   float64       
 4   Close    1144 non-null   float64       
 5   Volume   1144 non-null   float64       
 6   Change   1144 non-null   float64       
 7   title    1144 non-null   object        
 8   summary  1144 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 89.4+ KB


In [57]:
# api_key = "7A2qEj2N1XlxkYJogRqKFtkBWUkr32UQ"
# url = f"https://api.polygon.io/v2/aggs/ticker/{nyse}/range/1/minute/{start}/{end}?limit=50000&apiKey={api_key}"

# response = requests.get(url)
# data = response.json()
# if "results" in data:
#     df_temp = pd.DataFrame(data["results"])
#     df_temp["t"] = pd.to_datetime(df_temp["t"], unit="ms")
#     df_temp.rename(columns={"t":"Time","o": "Open", "h": "High", "l": "Low", "c": "Close", "v": "Volume"}, inplace=True)
#     df_temp["Change"] = df_temp["Close"] - df_temp["Open"]
#     df_temp = df_temp[["Time","Open", "High", "Low", "Close", "Volume", "Change"]]
#     df_news['Date'] = pd.to_datetime(df_news['Date'])
#     df_temp = pd.merge(df_temp, df_news, left_on="Time", right_on="Date", how="right")
#     df_temp = df_temp.drop(columns=["Date"])
#     #title과 summary가 NaN인 행을 삭제 한다.
#     df_temp = df_temp.dropna(subset=["title", "summary"])
# else:
#     print("Error: ", data["error"])
# # # csv로 저장
# df_temp.to_csv("AAPL_2025_right.csv", index=False)

In [58]:
# df_temp.info()

In [59]:
# df_upndown이라는 새로운 df를 만들어 해당 date에 close - open을 하여 이 값이 > 0 이면 up, < 0 이면 down, == 0 이면 same으로 df에 추가
df_upndown = df.copy()
df_upndown['upndown'] = df_upndown['Change'].apply(lambda x: 'up' if x > 0.1 else ('down' if x < -0.1 else 'same'))
df_upndown = df_upndown[["Time","Open", "High", "Low", "Close", "Volume", "Change",'upndown',"title", "summary"]]
df_upndown.head(10)

,Time,Open,High,Low,Close,Volume,Change,upndown,title,summary
14316,2025-01-02 09:32:00,251.75,251.83,251.7500,251.83,904.0,0.08,same,Jim Cramer's top 10 things to watch in the sto...,"On January 2, Wall Street started the new trad..."
14328,2025-01-02 10:12:00,250.67,250.81,250.6100,250.81,1683.0,0.14,up,Apple to pay $95 million to settle Siri privac...,Apple agreed to pay $95 million to resolve a p...
14329,2025-01-02 10:12:00,250.67,250.81,250.6100,250.81,1683.0,0.14,up,Apple to pay $95 million to settle Siri privac...,Apple has agreed to settle a class action laws...
14341,2025-01-02 10:30:00,250.79,250.79,250.7100,250.71,567.0,-0.08,same,CPSC Secures Agreement with Apple for Enhanced...,"On January 2, 2025, the U.S. Consumer Product ..."
14377,2025-01-02 11:39:00,249.96,249.96,249.9000,249.90,1641.0,-0.06,same,Apple's next Magic Mouse may support voice com...,Apple is reportedly developing a Magic Mouse p...
14393,2025-01-02 12:00:00,250.35,250.47,250.2500,250.25,8710.0,-0.10,same,"Top Stock Movers Now: Uber, Unity Software, Te...","On the first trading day of 2025, U.S. equitie..."
14447,2025-01-02 13:08:00,250.62,250.62,250.6200,250.62,676.0,0.00,same,Warren Buffett's Berkshire Hathaway beats S&P ...,"In 2024, Warren Buffett's Berkshire Hathaway o..."
14507,2025-01-02 14:12:00,248.92,249.03,248.8900,249.03,2525.0,0.11,up,Wall Street reverses early gains in first 2025...,The CNBC Investing Club with Jim Cramer provid...
14524,2025-01-02 14:29:00,248.83,248.95,248.5645,248.90,7356.0,0.07,same,Siri “unintentionally” recorded private convos...,Apple has agreed to pay $95 million to settle ...
14529,2025-01-02 14:34:00,246.75,246.94,246.3000,246.33,289896.0,-0.42,down,Apple to pay $95 million to settle Siri privac...,Apple has agreed to settle a class action laws...


In [60]:
def preproces_text(text):
    # 2. 축약어 확장
    text = contractions.fix(text)
    # 3. 특수 문자 제거
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # 4. 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    # 5. 소문자 변환
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\b\w{1,3}\b', '', text)
    
    return text

In [61]:
for index, row in df_upndown.iterrows():
    title = row['title']
    summary = row['summary']
    title = preproces_text(title)
    summary = preproces_text(summary)
    df_upndown.at[index, 'title'] = title
    df_upndown.at[index, 'summary'] = summary
df_upndown.to_csv("AAPL_preprocessed_1.csv", index=False)

In [62]:
df_upndown.head(10)

,Time,Open,High,Low,Close,Volume,Change,upndown,title,summary
14316,2025-01-02 09:32:00,251.75,251.83,251.7500,251.83,904.0,0.08,same,cramers things watch stock market thursday,january wall street started trading year p...
14328,2025-01-02 10:12:00,250.67,250.81,250.6100,250.81,1683.0,0.14,up,apple million settle siri privacy lawsuit,apple agreed million resolve proposed cla...
14329,2025-01-02 10:12:00,250.67,250.81,250.6100,250.81,1683.0,0.14,up,apple million settle siri privacy lawsuit,apple agreed settle class action lawsuit p...
14341,2025-01-02 10:30:00,250.79,250.79,250.7100,250.71,567.0,-0.08,same,cpsc secures agreement with apple enhanced wa...,january yous consumer product safety commi...
14377,2025-01-02 11:39:00,249.96,249.96,249.9000,249.90,1641.0,-0.06,same,apples next magic mouse support voice command...,apple reportedly developing magic mouse prot...
14393,2025-01-02 12:00:00,250.35,250.47,250.2500,250.25,8710.0,-0.10,same,stock movers uber unity software tesla more,first trading yous equities modest gains...
14447,2025-01-02 13:08:00,250.62,250.62,250.6200,250.62,676.0,0.00,same,warren buffetts berkshire hathaway beats p...,warren buffetts berkshire hathaway outperfor...
14507,2025-01-02 14:12:00,248.92,249.03,248.8900,249.03,2525.0,0.11,up,wall street reverses early gains first tradi...,cnbc investing club with cramer provides ho...
14524,2025-01-02 14:29:00,248.83,248.95,248.5645,248.90,7356.0,0.07,same,siri unintentionally recorded private convos a...,apple agreed million settle lawsuit clai...
14529,2025-01-02 14:34:00,246.75,246.94,246.3000,246.33,289896.0,-0.42,down,apple million settle siri privacy lawsuit,apple agreed settle class action lawsuit ...


In [63]:
nyse = 'AAPL'
end_date = datetime.strptime("2025-03-31", "%Y-%m-%d")
months_to_collect = 24
step_days = 30

df_all = pd.DataFrame()

for _ in range(months_to_collect):
    start_date = end_date - timedelta(days=step_days)

    # 2024년 3월 3일 이전부터는 수집하지 않음
    if start_date < datetime(2024, 3, 3):
        break

    start = start_date.strftime('%Y-%m-%d')
    end = end_date.strftime('%Y-%m-%d')

    for i in range(1, 46):
        url_news = f'https://api-v2.deepsearch.com/v1/global-articles?symbols=NYSE:{nyse}&page={i}&page_size=100&date_from={start}&date_to={end}&api_key=cffaa97f7c2f4ad0bd501ce7cd00b7e1'
        data = requests.get(url_news).json()
        if "data" not in data:
            break
        title_summary_list = [(item["title"], item["summary"], item["published_at"]) for item in data["data"]]
        df_news_temp = pd.DataFrame([
            {"Date": published_at, "title": title, "summary": summary}
            for title, summary, published_at in title_summary_list
        ])
        if i == 1:
            df_news = df_news_temp.copy()
        else:
            df_news = pd.concat([df_news, df_news_temp], ignore_index=True)

    df_news['Date'] = pd.to_datetime(df_news['Date']).dt.tz_convert('America/New_York')
    df_news['Date'] = df_news['Date'].dt.strftime('%Y-%m-%d %H:%M:00')

    url_stock = f"https://api.polygon.io/v2/aggs/ticker/{nyse}/range/1/minute/{start}/{end}?limit=50000&apiKey=7A2qEj2N1XlxkYJogRqKFtkBWUkr32UQ"
    response = requests.get(url_stock)
    data = response.json()
    if "results" in data:
        df_price = pd.DataFrame(data["results"])
        df_price["t"] = pd.to_datetime(df_price["t"], unit="ms")
        df_price.rename(columns={"t": "Time", "o": "Open", "h": "High", "l": "Low", "c": "Close", "v": "Volume"}, inplace=True)
        df_price["Change"] = df_price["Close"] - df_price["Open"]
        df_price = df_price[["Time", "Open", "High", "Low", "Close", "Volume", "Change"]]

        df_news['Date'] = pd.to_datetime(df_news['Date'])
        df_merged = pd.merge(df_price, df_news, left_on="Time", right_on="Date", how="left")
        df_merged = df_merged.drop(columns=["Date"])
        df_merged = df_merged.dropna(subset=["title", "summary"])

        df_all = pd.concat([df_all, df_merged], ignore_index=True)
    else:
        print(f"Stock API Error for {start} to {end}: ", data.get("error", "Unknown error"))

    end_date = start_date

df_all

,Time,Open,High,Low,Close,Volume,Change,title,summary
0,2025-03-03 09:01:00,241.480,241.570,241.410,241.5500,2115.0,0.0700,Apple (NASDAQ:AAPL) Earns Outperform Rating fr...,"Wedbush reaffirmed Apple's ""outperform"" rating..."
1,2025-03-03 09:15:00,241.470,241.470,241.380,241.3800,643.0,-0.0900,Warren Buffett Is Still Holding His Apple Stoc...,Warren Buffett has maintained Berkshire Hathaw...
2,2025-03-03 09:15:00,241.470,241.470,241.380,241.3800,643.0,-0.0900,Warren Buffett Is Still Holding His Apple Stoc...,Warren Buffett has maintained his Apple (NASDA...
3,2025-03-03 10:42:00,241.500,241.500,241.500,241.5000,1097.0,0.0000,Apple’s M4 MacBook Air refresh may be imminent...,Apple is set to continue its early 2025 produc...
4,2025-03-03 11:23:00,241.550,241.550,241.550,241.5500,297.0,0.0000,QUALCOMM Stock: Is QCOM Underperforming the Te...,"QUALCOMM Incorporated (QCOM), based in San Die..."
...,...,...,...,...,...,...,...,...,...
7512,2024-04-05 16:13:00,170.145,170.175,170.135,170.1350,39015.0,-0.0100,Beats headphones normally worth $350 have pric...,Walmart has reduced the price of wireless Beat...
7513,2024-04-05 16:16:00,170.250,170.255,170.195,170.2250,40886.0,-0.0250,The best tech to have in a natural disaster,"Smartphones are not just for entertainment, th..."
7514,2024-04-05 16:29:00,170.095,170.175,170.095,170.1700,39753.0,0.0750,Fans return to Bonnie Tyler's 'Total Eclipse o...,"Bonnie Tyler's 1983 hit ""Total Eclipse of the ..."
7515,2024-04-05 17:38:00,170.090,170.120,170.050,170.0550,36273.0,-0.0350,Ex-Yahoo CEO Marissa Mayer’s new app has just ...,Former Silicon Valley star Marissa Mayer's new...


In [64]:
df_all.to_csv("AAPL_7500.csv", index=False)

In [65]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7517 entries, 0 to 7516
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Time     7517 non-null   datetime64[ns]
 1   Open     7517 non-null   float64       
 2   High     7517 non-null   float64       
 3   Low      7517 non-null   float64       
 4   Close    7517 non-null   float64       
 5   Volume   7517 non-null   float64       
 6   Change   7517 non-null   float64       
 7   title    7517 non-null   object        
 8   summary  7517 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 528.7+ KB


In [66]:
# df_7400이라는 새로운 df를 만들어 해당 date에 close - open을 하여 이 값이 > 0 이면 up, < 0 이면 down, == 0 이면 same으로 df에 추가
df_7400 = df_all.copy()
df_7400['upndown'] = df_7400['Change'].apply(lambda x: 'up' if x > 0 else ('down' if x < 0 else 'same'))
df_7400 = df_7400[["Time","Open", "High", "Low", "Close", "Volume", "Change",'upndown',"title", "summary"]]
df_7400.head(10)
for index, row in df_7400.iterrows():
    title = row['title']
    summary = row['summary']
    title = preproces_text(title)
    summary = preproces_text(summary)
    df_7400.at[index, 'title'] = title
    df_7400.at[index, 'summary'] = summary
    #df_7400에서 upndown의 up의 갯수와 down의 갯수와 same의 갯수를 동일하게 맞춘다.
df_7400_up = df_7400[df_7400['upndown'] == 'up']
df_7400_down = df_7400[df_7400['upndown'] == 'down']
df_7400_same = df_7400[df_7400['upndown'] == 'same']
# 클래스별 개수 파악
count_up = len(df_7400_up)
count_down = len(df_7400_down)
count_same = len(df_7400_same)

# 최소 클래스 수를 기준으로 샘플 수 결정
min_count = min(count_up, count_down, count_same)

# 각 클래스에서 동일한 수만큼 샘플링
df_7400_up = df_7400_up.sample(n=min_count, random_state=42)
df_7400_down = df_7400_down.sample(n=min_count, random_state=42)
df_7400_same = df_7400_same.sample(n=min_count, random_state=42)

# 데이터프레임 통합 및 셔플
df_7400 = pd.concat([df_7400_up, df_7400_down, df_7400_same], ignore_index=True)
df_7400 = df_7400.sample(frac=1, random_state=42).reset_index(drop=True)


df_7400.to_csv("AAPL_7400_preprocessing.csv", index=False)
df_7400

,Time,Open,High,Low,Close,Volume,Change,upndown,title,summary
0,2024-10-29 18:28:00,233.9800,234.000,233.9350,233.98,31865.0,0.0000,same,asia muted open after mixed tech results mar...,asian stocks expected open cautiously follow...
1,2024-08-09 14:31:00,214.4731,214.495,214.1700,214.21,98445.0,-0.2631,down,lillys earnings rally done plus stayed ...,cnbc investing club with cramer provides da...
2,2024-05-13 16:06:00,186.2100,186.290,186.1814,186.21,279018.0,0.0000,same,what implications partnership between opena...,apple close striking deal with openai brin...
3,2024-09-10 10:20:00,218.0600,218.060,218.0600,218.06,543.0,0.0000,same,when iphone release date apples powerhouse...,apple announced iphone series comprising fo...
4,2024-06-11 09:30:00,191.5100,191.510,191.5100,191.51,341.0,0.0000,same,stock market today stocks waver near records ...,stocks were mixed tuesday investors awaited...
...,...,...,...,...,...,...,...,...,...,...
4285,2024-06-26 11:55:00,211.0300,211.030,211.0300,211.03,586.0,0.0000,same,instruction manuals memoriam,instruction manuals have become rare leading ...
4286,2025-03-12 23:53:00,217.2500,217.380,217.2500,217.38,833.0,0.1300,up,regulator apple google reason innovation ...,competition markets authority determined ...
4287,2024-06-14 09:00:00,214.2300,214.230,214.2300,214.23,396.0,0.0000,same,apple king back race trillion,apple aapl shares have broken through trill...
4288,2025-01-29 12:36:00,235.5800,235.580,235.5800,235.58,999.0,0.0000,same,navy email warns against using apps like chi...,navy issued internal memo advising members ...


In [67]:
df_7400.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4290 entries, 0 to 4289
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Time     4290 non-null   datetime64[ns]
 1   Open     4290 non-null   float64       
 2   High     4290 non-null   float64       
 3   Low      4290 non-null   float64       
 4   Close    4290 non-null   float64       
 5   Volume   4290 non-null   float64       
 6   Change   4290 non-null   float64       
 7   upndown  4290 non-null   object        
 8   title    4290 non-null   object        
 9   summary  4290 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(3)
memory usage: 335.3+ KB


In [68]:
# 🔧 2. 데이터 로드 및 준비
df = df_7400.copy()
df['text'] = df['title'].astype(str) + " " + df['summary'].astype(str)

le = LabelEncoder()
df['label'] = le.fit_transform(df['upndown'])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# 🔧 3. 토크나이저 정의
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# ✅ 수정된 Dataset 클래스
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label)
        }

# ✅ 수정된 Dataset 인스턴스
train_dataset = NewsDataset(train_texts, train_labels, tokenizer)
val_dataset = NewsDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# 🔧 4. 모델 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
model.to(device)

# 🔧 5. 학습 루프
optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

# 🔧 6. 모델 저장
model.save_pretrained("bert_news_4_model")
tokenizer.save_pretrained("bert_news_4_model")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 215/215 [10:11<00:00,  2.84s/it]


Epoch 1 Loss: 238.3245


Epoch 2: 100%|██████████| 215/215 [10:10<00:00,  2.84s/it]


Epoch 2 Loss: 237.0821


Epoch 3: 100%|██████████| 215/215 [10:25<00:00,  2.91s/it]


Epoch 3 Loss: 238.9546


Epoch 4: 100%|██████████| 215/215 [10:17<00:00,  2.87s/it]


Epoch 4 Loss: 238.0782


('bert_news_4_model/tokenizer_config.json',
 'bert_news_4_model/special_tokens_map.json',
 'bert_news_4_model/vocab.txt',
 'bert_news_4_model/added_tokens.json')

In [69]:
nyse = 'AAPL'
start = '2025-05-01'
end = '2025-05-01'

for i in range (1, 2):
    if(i == 1):
        url = 'https://api-v2.deepsearch.com/v1/global-articles?symbols=NYSE:'+nyse+'&page='+str(i)+'&page_size=100&date_from='+start+'&date_to='+end+'&api_key=cffaa97f7c2f4ad0bd501ce7cd00b7e1'
        data = requests.get(url).json()
        title_summary_list = [(item["title"], item["summary"], item["published_at"]) for item in data["data"]]
        df_news = pd.DataFrame([
            {"Date": published_at, "title": title, "summary": summary}
            for title, summary, published_at in title_summary_list
        ])
    else:
        url = 'https://api-v2.deepsearch.com/v1/global-articles?symbols=NYSE:'+nyse+'&page='+str(i)+'&page_size=100&date_from='+start+'&date_to='+end+'&api_key=cffaa97f7c2f4ad0bd501ce7cd00b7e1'
        data = requests.get(url).json()
        title_summary_list = [(item["title"], item["summary"], item["published_at"]) for item in data["data"]]
        df_news_temp = pd.DataFrame([
            {"Date": published_at, "title": title, "summary": summary}
            for title, summary, published_at in title_summary_list
        ])
        df_news = pd.concat([df_news, df_news_temp], ignore_index=True)

df_news['Date'] = pd.to_datetime(df_news['Date']).dt.tz_convert('America/New_York')
df_news['Date'] = df_news['Date'].dt.strftime('%Y-%m-%d %H:%M:00')


api_key = "7A2qEj2N1XlxkYJogRqKFtkBWUkr32UQ"
url = f"https://api.polygon.io/v2/aggs/ticker/{nyse}/range/1/minute/{start}/{end}?limit=50000&apiKey={api_key}"

response = requests.get(url)
data = response.json()
if "results" in data:
    df = pd.DataFrame(data["results"])
    df["t"] = pd.to_datetime(df["t"], unit="ms")
    df.rename(columns={"t":"Time","o": "Open", "h": "High", "l": "Low", "c": "Close", "v": "Volume"}, inplace=True)
    df["Change"] = df["Close"] - df["Open"]
    df = df[["Time","Open", "High", "Low", "Close", "Volume", "Change"]]
    df_news['Date'] = pd.to_datetime(df_news['Date'])
    df = pd.merge(df, df_news, left_on="Time", right_on="Date", how="left")
    df = df.drop(columns=["Date"])
    #title과 summary가 NaN인 행을 삭제 한다.
    df = df.dropna(subset=["title", "summary"])
else:
    print("Error: ", data["error"])

# df_upndown이라는 새로운 df를 만들어 해당 date에 close - open을 하여 이 값이 > 0 이면 up, < 0 이면 down, == 0 이면 same으로 df에 추가
df_upndown = df.copy()
df_upndown['upndown'] = df_upndown['Change'].apply(lambda x: 'up' if x > 0.1 else ('down' if x < -0.1 else 'same'))
df_upndown = df_upndown[["Time","Open", "High", "Low", "Close", "Volume", "Change",'upndown',"title", "summary"]]
df_upndown.to_csv("AAPL_2025_test.csv", index=False)

In [70]:
# 1. 모델과 토크나이저 불러오기
model = BertForSequenceClassification.from_pretrained("bert_news_model")
tokenizer = BertTokenizer.from_pretrained("bert_news_model")
model.eval()

# 2. 인코딩 + 예측 함수
def predict_news_sentiment(title, summary):
    text = title + " " + summary
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    # 3. 숫자 → 라벨 디코딩
    label_map = {0: 'up', 1: 'down', 2: 'same'}
    return label_map[pred], probs.tolist()[0]

#df_upndown에 있는 title과 summary를 넣어 예측 후 prediction이라는 곳에 up 혹은 down 혹은 same을 넣는다.
predictions = []
for index, row in df_upndown.iterrows():
    title = row['title']
    summary = row['summary']
    pred, probs = predict_news_sentiment(title, summary)
    predictions.append(pred)
df_upndown['prediction'] = predictions
df_upndown['prediction'] = df_upndown['prediction'].astype(str)

#df_upndown에 있는 upndown과 prediction을 비교하여 맞으면 1, 틀리면 0을 넣는다.
df_upndown['correct'] = df_upndown.apply(lambda x: 1 if x['upndown'] == x['prediction'] else 0, axis=1)
df_upndown['correct'] = df_upndown['correct'].astype(int)
df_upndown.to_csv("AAPL_2025_test_final.csv", index=False)

In [71]:
#df_upndown에 있는 correct의 1과 0의 갯수를 구하고 출력
correct_count = df_upndown['correct'].value_counts()
print("정확도: ", correct_count[1])
print("오답: ", correct_count[0])
# 정확도 = 맞은 개수 / 전체 개수
accuracy = correct_count[1] / (correct_count[1] + correct_count[0])
print(f"정확도 비율: {accuracy:.2%}")

정확도:  35
오답:  19
정확도 비율: 64.81%
